## Week 3

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df1_animation = pd.read_csv("data/Animation_Movies_clean.csv")

In [3]:
df1_animation.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'budget', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'genres_clean'],
      dtype='str')

In [6]:
df1_animation["popularity"] = df1_animation["popularity"].replace(
    0,
    np.nan
)

In [7]:
(df1_animation["popularity"] <= 0).sum()

np.int64(0)

In [8]:
df1_animation["log_budget"] = np.log10(
    df1_animation["budget"]
)

df1_animation["log_revenue"] = np.log10(
    df1_animation["revenue"]
)

df1_animation["log_popularity"] = np.log10(
    df1_animation["popularity"]
)

In [9]:
animation_week3 = df1_animation.dropna(
    subset=[
        "log_budget",
        "log_popularity",
        "vote_average",
        "vote_count",
        "runtime",
        "log_revenue"
    ]
).copy()

animation_week3.shape

(616, 23)

In [10]:
X = animation_week3[
    [
        "log_budget",
        "log_popularity",
        "vote_average",
        "vote_count",
        "runtime"
    ]
]

y = animation_week3["log_revenue"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 492
Testing rows: 124


### Forward Selection 

In [14]:
remaining = list(X_train.columns)
selected = []

while remaining:

    scores = []

    for feature in remaining:

        features = selected + [feature]

        model = LinearRegression()

        model.fit(
            X_train[features],
            y_train
        )

        r2 = model.score(
            X_train[features],
            y_train
        )

        scores.append((r2, feature))

    best_r2, best_feature = max(scores)

    selected.append(best_feature)
    remaining.remove(best_feature)

    print(
        "Added:",
        best_feature,
        "| R2:",
        round(best_r2, 4)
    )

Added: log_budget | R2: 0.8672
Added: log_popularity | R2: 0.8903
Added: vote_count | R2: 0.8956
Added: vote_average | R2: 0.896
Added: runtime | R2: 0.896


In [15]:
selected

['log_budget', 'log_popularity', 'vote_count', 'vote_average', 'runtime']

### Backward 

In [16]:
backward_features = list(X_train.columns)

model = LinearRegression()

model.fit(
    X_train[backward_features],
    y_train
)

full_r2 = model.score(
    X_train[backward_features],
    y_train
)

print("Starting features:", backward_features)
print("Full model R2:", full_r2)

Starting features: ['log_budget', 'log_popularity', 'vote_average', 'vote_count', 'runtime']
Full model R2: 0.8960086465059935


In [17]:
features = list(X_train.columns)

while len(features) > 1:

    scores = []

    for feature in features:

        remaining_features = [
            f for f in features if f != feature
        ]

        model = LinearRegression()

        model.fit(
            X_train[remaining_features],
            y_train
        )

        r2 = model.score(
            X_train[remaining_features],
            y_train
        )

        scores.append((r2, feature))

    best_r2, feature_to_remove = max(scores)

    features.remove(feature_to_remove)

    print(
        "Removed:",
        feature_to_remove,
        "| R2:",
        round(best_r2, 4)
    )

Removed: runtime | R2: 0.896
Removed: vote_average | R2: 0.8956
Removed: vote_count | R2: 0.8903
Removed: log_popularity | R2: 0.8672


In [18]:
features

['log_budget']

### PCR

In [19]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [20]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
pca = PCA()

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [22]:
pca.explained_variance_ratio_

array([0.64278819, 0.14776507, 0.12933459, 0.04611041, 0.03400175])

In [23]:
np.cumsum(pca.explained_variance_ratio_)

array([0.64278819, 0.79055325, 0.91988784, 0.96599825, 1.        ])

In [24]:
X_train_pcr = X_train_pca[:, :3]
X_test_pcr = X_test_pca[:, :3]

In [25]:
pcr_model = LinearRegression()

pcr_model.fit(
    X_train_pcr,
    y_train
)

y_pred_pcr = pcr_model.predict(
    X_test_pcr
)

In [26]:
pcr_r2 = r2_score(
    y_test,
    y_pred_pcr
)

pcr_rmse = mean_squared_error(
    y_test,
    y_pred_pcr
) ** 0.5

print("PCR R2:", pcr_r2)
print("PCR RMSE:", pcr_rmse)

PCR R2: 0.8628014213887036
PCR RMSE: 0.702698344069031


### PLSR

In [27]:
from sklearn.cross_decomposition import PLSRegression

In [28]:
pls_model = PLSRegression(
    n_components=3
)

pls_model.fit(
    X_train_scaled,
    y_train
)

y_pred_pls = pls_model.predict(
    X_test_scaled
).ravel()

In [29]:
pls_r2 = r2_score(
    y_test,
    y_pred_pls
)

pls_rmse = mean_squared_error(
    y_test,
    y_pred_pls
) ** 0.5

print("PLSR R2:", pls_r2)
print("PLSR RMSE:", pls_rmse)

PLSR R2: 0.9092381764057793
PLSR RMSE: 0.5715388158277032


### Dataframe 2

In [30]:
df2_action = pd.read_csv("data/Action_Movies_clean.csv")

In [31]:
df2_action.columns

Index(['movie_id', 'movie_name', 'year', 'runtime', 'genre', 'rating',
       'description', 'director', 'director_id', 'star', 'star_id', 'votes',
       'gross(in $)', 'genre_clean'],
      dtype='str')

In [32]:
(df2_action["votes"] <= 0).sum()

np.int64(0)

In [33]:
(df2_action["gross(in $)"] <= 0).sum()

np.int64(0)

In [34]:
df2_action["log_votes"] = np.log10(df2_action["votes"])
df2_action["log_gross"] = np.log10(df2_action["gross(in $)"])

In [35]:
action_week3 = df2_action.dropna(
    subset=[
        "log_votes",
        "rating",
        "runtime",
        "year",
        "log_gross"
    ]
).copy()

action_week3.shape

(2758, 16)

### Forward Selection

In [36]:
features = [
    "log_votes",
    "rating",
    "runtime",
    "year"
]

X = action_week3[features]
y = action_week3["log_gross"]

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [38]:
remaining = features.copy()
selected = []

while remaining:
    best_feature = None
    best_r2 = -np.inf

    for feature in remaining:
        test_features = selected + [feature]

        model = LinearRegression()
        model.fit(X_train[test_features], y_train)

        r2 = model.score(
            X_test[test_features],
            y_test
        )

        if r2 > best_r2:
            best_r2 = r2
            best_feature = feature

    selected.append(best_feature)
    remaining.remove(best_feature)

    print(
        "Added:", best_feature,
        "| R2:", round(best_r2, 4)
    )

Added: log_votes | R2: 0.5484
Added: year | R2: 0.5997
Added: rating | R2: 0.615
Added: runtime | R2: 0.617


### Backward

In [39]:
features = [
    "log_votes",
    "rating",
    "runtime",
    "year"
]

print("Starting features:", features)

model = LinearRegression()
model.fit(X_train[features], y_train)

full_r2 = model.score(
    X_test[features],
    y_test
)

print("Full model R2:", full_r2)

Starting features: ['log_votes', 'rating', 'runtime', 'year']
Full model R2: 0.617044929224712


In [40]:
features = [
    "log_votes",
    "rating",
    "runtime",
    "year"
]

while len(features) > 1:

    best_r2 = -np.inf
    feature_to_remove = None

    for feature in features:

        test_features = [
            f for f in features
            if f != feature
        ]

        model = LinearRegression()
        model.fit(
            X_train[test_features],
            y_train
        )

        r2 = model.score(
            X_test[test_features],
            y_test
        )

        if r2 > best_r2:
            best_r2 = r2
            feature_to_remove = feature

    features.remove(feature_to_remove)

    print(
        "Removed:", feature_to_remove,
        "| R2:", round(best_r2, 4)
    )

Removed: runtime | R2: 0.615
Removed: rating | R2: 0.5997
Removed: year | R2: 0.5484


In [41]:
features

['log_votes']

### PCR

In [42]:
features = [
    "log_votes",
    "rating",
    "runtime",
    "year"
]

X = action_week3[features]
y = action_week3["log_gross"]

In [43]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [44]:
pca.explained_variance_ratio_

array([0.46473185, 0.24940218, 0.18060416, 0.1052618 ])

In [45]:
np.cumsum(pca.explained_variance_ratio_)

array([0.46473185, 0.71413404, 0.8947382 , 1.        ])

In [46]:
X = action_week3[
    ["log_votes", "rating", "runtime", "year"]
]

y = action_week3["log_gross"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [47]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [48]:
pca = PCA(n_components=3)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [49]:
pcr_model = LinearRegression()

pcr_model.fit(
    X_train_pca,
    y_train
)

y_pred_pcr = pcr_model.predict(X_test_pca)

In [50]:
pcr_r2 = r2_score(
    y_test,
    y_pred_pcr
)

pcr_rmse = mean_squared_error(
    y_test,
    y_pred_pcr
) ** 0.5

print("PCR R2:", pcr_r2)
print("PCR RMSE:", pcr_rmse)

PCR R2: 0.3585963317325127
PCR RMSE: 1.0444940831388339


### PLSR

In [51]:
plsr_model = PLSRegression(n_components=3)

plsr_model.fit(
    X_train_scaled,
    y_train
)

y_pred_plsr = plsr_model.predict(
    X_test_scaled
).ravel()

In [52]:
plsr_r2 = r2_score(
    y_test,
    y_pred_plsr
)

plsr_rmse = mean_squared_error(
    y_test,
    y_pred_plsr
) ** 0.5

print("PLSR R2:", plsr_r2)
print("PLSR RMSE:", plsr_rmse)

PLSR R2: 0.615064688847993
PLSR RMSE: 0.8091595056534271
